# 01 — Exploratory Data Analysis

**Decoding Customer Value: A SQL-Driven Retention Strategy**

This notebook is intentionally thin: it *calls the production modules* in `src/` rather than re-implementing logic, so anything proven here is exactly what runs in the pipeline. Use it to explore; promote findings into `src/`.

It documents the three structural findings that shaped the whole project:
1. `Discount Applied` and `Promo Code Used` are perfectly collinear.
2. Every subscriber is discounted; every discounted customer is male (confounds).
3. No timestamps / churn / loyalty labels — these are engineered.

In [ ]:
import sys; sys.path.append('..')
import pandas as pd
from config.settings import get_settings
from src.utils.io import read_csv
from src.data_processing.clean import clean_dataset
from src.data_processing.profile import profile_dataframe

settings = get_settings()
raw = read_csv(settings.raw_data_file)
raw.shape

## 1. Structural integrity checks

In [ ]:
print('Discount == Promo (identical):', (raw['Discount Applied'] == raw['Promo Code Used']).all())
print('\nSubscription x Discount:')
print(pd.crosstab(raw['Subscription Status'], raw['Discount Applied']))
print('\nGender x Discount:')
print(pd.crosstab(raw['Gender'], raw['Discount Applied']))

**Read-out.** The discount column is redundant (dropped in feature engineering). Subscription and gender are deterministically entangled with discounting — so any 'subscribers/regions are discount-dependent' claim is a data artefact, not behaviour. The pipeline encodes these caveats in `src/business_logic/insights.py`.

## 2. Clean + profile

In [ ]:
clean = clean_dataset(raw)
prof = profile_dataframe(clean)
pd.DataFrame(prof['numeric_summary']).T

## 3. Engineered features + loyalty A/B

In [ ]:
from src.feature_engineering.build_features import build_features
from src.business_logic.loyalty_framework import evaluate_loyalty_definitions

feat = build_features(clean)
fw = evaluate_loyalty_definitions(feat)
print('Winner:', fw.winner)
for s in fw.scores:
    print(f'{s.label:32s} total={s.total:.3f}  {s.criteria}')

In [ ]:
ax = feat[['loyalty_score_v1','loyalty_score_v2']].plot(kind='hist', bins=30, alpha=0.6,
          title='Loyalty distribution: Definition A (v1) vs B (v2)')
ax.set_xlabel('loyalty score')

## 4. Value concentration (the 80/20 thesis)
The chart below is the single most important business fact: a small fraction of customers carries the majority of projected value.

In [ ]:
s = feat['estimated_clv_proxy'].sort_values(ascending=False).reset_index(drop=True)
cum = s.cumsum() / s.sum()
pct_customers = (cum.index + 1) / len(cum)
ax = pd.Series(cum.values, index=pct_customers).plot(
    title='Cumulative CLV vs cumulative customers (Lorenz-style)')
ax.set_xlabel('cumulative share of customers'); ax.set_ylabel('cumulative share of CLV')
print('Top 20% of customers hold {:.1%} of projected CLV'.format(
    cum.iloc[int(0.2*len(cum))]))

## Next steps
Run the full pipeline to materialise the SQL layer and reports:
```bash
python -m src.pipeline
```
Then explore interactively: `streamlit run src/dashboard/app.py`.